# Direct UPT boxes on 130 VALID images

Attach csa-scst-full-karpathy-cache-v1, csa-prompt-full-karpathy-v175, upt-prompts-full-all-actions-v2 and mscoco-2014. Use one GPU T4 and Internet. Enable Secret codex for v94, or set CHECKPOINT_PATH to the exact mounted update1500 checkpoint. Send csa_relation_direct_upt_valid130_review.zip.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
SOURCE_VERSION='ducanh3/notebook7bc83c4c76/94'
SECRET_NAME='codex'
CHECKPOINT_PATH=''  # Optional exact v94 pilot_checkpoint.bin mounted from authorized Output/Dataset
RESUME_SHA='80e1280b9eef216c5db421a738c7a5e0550bb34b1a62a2a13b2c8e76d40b1a24'
WALL_HOURS=3 # STOP cap including CPU metrics, not ETA or billing.
OUTPUT=Path('/kaggle/working/csa_relation_direct_upt_valid130')
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
TRAIN_REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
TRAIN_PINS={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}
CACHE_ROOT='/kaggle/input/datasets/ducanh3/csa-scst-full-karpathy-cache-v1'
if OUTPUT.exists():raise RuntimeError('Preserve existing captions/output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for TEST patches/checkpoint/CPU scorer assets')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch199-test-',dir='/tmp'));DEPS=SOURCE/'deps';CLI=SOURCE/'cli';CHAIR=SOURCE/'chair';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)


SAVED={4375: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000004375.jpg', 'image_sha256': '999c2ccf98b2798b57175a0e75d72fe856b5d05d8d1f8d3f992ec79f04a2d71d', 'detected': {'person': 0.7844}, 'caption': 'a group of people riding skis on a snow covered slope'}, 5282: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000005282.jpg', 'image_sha256': '2d286da0df7544038e01c2bc2dae4095e8f92ae322b3df388a3a7fbface56659', 'detected': {'boat': 0.8271, 'person': 0.5524, 'car': 0.5314}, 'caption': 'a blue and white boat traveling under a bridge'}, 13004: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000013004.jpg', 'image_sha256': '09d489992329d2a17186a82781c7d23d8c687ee580567babc7494f9bc8b1220a', 'detected': {}, 'caption': 'a banana is on a plate with a cup'}, 13031: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000013031.jpg', 'image_sha256': 'e9e8d3bb69e8f82f4dfd952e8b423f0cdbf4e3a1ef4f7f9ac78df4e9dba8e7a8', 'detected': {'boat': 0.8439, 'person': 0.6778}, 'caption': 'a man flying through the air while riding a board'}, 15303: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000015303.jpg', 'image_sha256': '0937e550afd880c8478e0b3667317ffffaae53329e8b3be532b7e6ebd559c9d9', 'detected': {'person': 0.8874, 'surfboard': 0.5934}, 'caption': 'a man riding a wave on top of a surfboard'}, 16546: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000016546.jpg', 'image_sha256': 'a99ba4724dcd00b137d577d25cb54ed87cc52ac823375542abaf0e946315c520', 'detected': {'person': 0.8946}, 'caption': 'two people cross country skiing in the snow'}, 18193: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000018193.jpg', 'image_sha256': 'ee9037876bab7f2b504cd4be49144bbde27f32ddc05361cd6f3c843c947dd911', 'detected': {'person': 0.9351, 'sandwich': 0.5045}, 'caption': 'a woman sitting in a chair eating food'}, 22420: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000022420.jpg', 'image_sha256': 'c07333f38a6fb52a3816e06cc21be3ad0251547215567a1398e726dc46b2c53d', 'detected': {'book': 0.9633, 'motorcycle': 0.6941}, 'caption': 'a couple of books sitting on top of a book shelf'}, 25005: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000025005.jpg', 'image_sha256': '4503c81279480d339c13f79d64a63609fc50455605a3956d332965c140a80658', 'detected': {'person': 0.9408, 'skis': 0.7336}, 'caption': 'a man riding skis across snow covered ground'}, 29573: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000029573.jpg', 'image_sha256': '7da3cca824b4cddd6212479479d4e62ed03329e9267096aab3d211b4a7604f29', 'detected': {'person': 0.9231, 'tie': 0.5954}, 'caption': 'a man in a suit walking in front of a building'}, 30347: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000030347.jpg', 'image_sha256': '1136e04bdc8a729ba237ee8058ee5678bf4ade1d80ab134285d987b3b9e9b5d4', 'detected': {'person': 0.9523, 'laptop': 0.8665, 'couch': 0.6988, 'chair': 0.6964, 'remote': 0.5425, 'refrigerator': 0.5131}, 'caption': 'a group of people standing around a laptop'}, 31113: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000031113.jpg', 'image_sha256': '11442787d7c45731d5c6fe3def1cbe9f0bea58bf3f164d3659c61f01aeee3c33', 'detected': {'person': 0.9446}, 'caption': 'a boy standing in a room with a microphone'}, 36032: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000036032.jpg', 'image_sha256': '092b57fce620db239981b40e4715aea4e6d1d5b005090b9fc6c6a2e6dda1556d', 'detected': {'person': 0.8738}, 'caption': 'a man riding a snowboard down a snow covered slope'}, 36607: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000036607.jpg', 'image_sha256': 'fd2a17b65c84cb5795a825499aa10596b8658b1c8dc590ee040acaf70a15a640', 'detected': {'cell phone': 0.9323, 'person': 0.694}, 'caption': 'a man holding a cell phone in front of a door'}, 46101: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000046101.jpg', 'image_sha256': '56b5591f8885834d719f9644c67ca429b1d6eed6d122ef268f46df6668f4c15f', 'detected': {'person': 0.8975, 'chair': 0.8361, 'tennis racket': 0.803, 'backpack': 0.6314}, 'caption': 'a couple of women standing on top of a tennis court'}, 48554: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000048554.jpg', 'image_sha256': '859837b4332079677cd95f952be05bd1026488cb165772093427af0ee78d3c4a', 'detected': {'person': 0.8929, 'skateboard': 0.7524, 'sports ball': 0.5981}, 'caption': 'a group of kids playing a game of basketball'}, 55167: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000055167.jpg', 'image_sha256': '370ff0c32b47f0200f7a0d2bfa507f2684d24a29fbfde06f75ed354910ac371b', 'detected': {'bus': 0.95, 'car': 0.7421, 'person': 0.7038, 'truck': 0.5997}, 'caption': 'a yellow and blue bus driving down a street next to a mountain'}, 57265: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000057265.jpg', 'image_sha256': '7f7156c13ef7fc450bb19ca960f5ad8597b59033535e161ddc713f0de1a6e91c', 'detected': {'person': 0.9252, 'handbag': 0.8643, 'umbrella': 0.5613}, 'caption': 'a man and a woman standing under an umbrella'}, 58225: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000058225.jpg', 'image_sha256': '9fc7e1e2fd37aec58136d4cda553f666285996491b78e16a54306891b6926ef3', 'detected': {'person': 0.9249, 'hot dog': 0.871}, 'caption': 'a person holding a hot dog in front of a large stadium'}, 63549: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000063549.jpg', 'image_sha256': '68e700a4a5c7c3824b68d6c1a848b5f26f2615a3f7911b145b51ec5c9e323212', 'detected': {'skis': 0.9401, 'person': 0.91}, 'caption': 'a man standing on top of a snow covered slope'}, 66485: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000066485.jpg', 'image_sha256': '8ed03723b508024df26495f3064a2533bd521c8b8f88588a6c471b7317ed3437', 'detected': {'person': 0.9175}, 'caption': 'a group of people playing a game of frisbee'}, 73927: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000073927.jpg', 'image_sha256': '47a9ebf3343822f0a0b1cf985b54e0ba4de9cc67e07aa8a13f1e2467aac99d49', 'detected': {'person': 0.9364, 'baseball glove': 0.6945, 'baseball bat': 0.6306}, 'caption': 'a baseball player holding a bat on a field'}, 76822: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000076822.jpg', 'image_sha256': '5c06963277a901764d8dfc2bbf480b28364aa9c0226c9906cd6c0480dd65b262', 'detected': {'person': 0.8525, 'skis': 0.6983}, 'caption': 'a person flying through the air on skis'}, 80895: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000080895.jpg', 'image_sha256': 'ccccd327374176e5b9867eb5502e306c17abb1c082e330dbefbbf3ded6e612db', 'detected': {'person': 0.9403, 'cell phone': 0.7348, 'boat': 0.6232}, 'caption': 'a man and a woman looking at their cell phones'}, 82396: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000082396.jpg', 'image_sha256': '024cc26e4dfa6f09b77e16d383eedee25203f315a10b2172472b075f6b2ce96f', 'detected': {'horse': 0.9412, 'person': 0.7767}, 'caption': 'a statue of a person on a horse'}, 82986: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000082986.jpg', 'image_sha256': 'e7f137f0d3721b27156871278f6f1b9309541444d2dc8095f109e46cf013e8aa', 'detected': {'person': 0.8933, 'frisbee': 0.77}, 'caption': 'a group of men playing a game of frisbee'}, 83573: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000083573.jpg', 'image_sha256': '87745bbc118baf63e8c4bd2463b132ce858cc426eb2bf8a9a8c45f7a3a38bc4d', 'detected': {'person': 0.894, 'backpack': 0.7246, 'skis': 0.6594}, 'caption': 'a group of people standing on top of a snow covered slope'}, 83925: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000083925.jpg', 'image_sha256': '664127758067128e4b2be61fea490a3d214fe96dcdaa7eac81263bd0ef6159a9', 'detected': {'person': 0.9342, 'sports ball': 0.8435, 'baseball bat': 0.7222}, 'caption': 'a young boy hitting a ball with a tennis racquet'}, 94607: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000094607.jpg', 'image_sha256': 'cbb826c99825e4a95198d749982a26018a51b3eb65047be03836dfb8be995b7d', 'detected': {'person': 0.8873, 'donut': 0.8089}, 'caption': 'a man holding a cup of coffee and a doughnut'}, 105973: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000105973.jpg', 'image_sha256': 'fbefed19b6fec652fd726cfde70d703535e968b724af8d25701a605908e1bccd', 'detected': {'person': 0.929, 'baseball bat': 0.6336}, 'caption': 'a baseball player holding a baseball bat on a field'}, 111909: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000111909.jpg', 'image_sha256': '4e24800169f9be5f4e7e7de8608dd72ee0420747c84f5a400738270cdf8b99ad', 'detected': {'person': 0.8773}, 'caption': 'a man water skiing behind a boat in the water'}, 118740: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000118740.jpg', 'image_sha256': '206c26c3e3eb2756371e39da28b4f3eb91aa7f53c053580e0fa88cefeacf36c1', 'detected': {'person': 0.8997}, 'caption': 'a man riding a wave on top of a surfboard'}, 122542: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000122542.jpg', 'image_sha256': '622083b6ed4285f7e32571d9bc660ecaef750412ec09a8f75ecef1c132412b46', 'detected': {'person': 0.9096, 'skis': 0.7387}, 'caption': 'a man and woman standing on skis in the snow'}, 122766: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000122766.jpg', 'image_sha256': '094754a4664099cc7a6fc12465d18c840c666d38104c15396ccba161a7e0247b', 'detected': {'person': 0.8997, 'skateboard': 0.8083}, 'caption': 'a child riding a skateboard on a street'}, 124116: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000124116.jpg', 'image_sha256': 'b519d4dca23c99921bbd132a525a65acdd8ea39464684b72096b009354663487', 'detected': {'cat': 0.9158, 'suitcase': 0.8892}, 'caption': 'a white and gray cat laying in a suitcase'}, 131869: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000131869.jpg', 'image_sha256': 'f8145242fc1fd0e9f64fc4abf31bdf3d84e464367968a2a093a22fdd2787d323', 'detected': {'person': 0.9086}, 'caption': 'a man standing on a tennis court holding a racquet'}, 133620: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000133620.jpg', 'image_sha256': 'c8655702bd287384bb2e89645e1e3f781b412a0dfacf7e112c59e1c3c48d2646', 'detected': {'person': 0.944, 'tie': 0.6465}, 'caption': 'a woman in a uniform standing next to a door'}, 134103: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000134103.jpg', 'image_sha256': '8068bb1133bd1481ef858a6b964272381e0870c7927409bdebb2f0c9c6a3be8b', 'detected': {'horse': 0.8925, 'person': 0.8235}, 'caption': 'a group of people riding horses across a snow covered field'}, 135155: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000135155.jpg', 'image_sha256': '077d6ea1f51c110ef8429f5dadeef45213a513fb7f2b9c4d5e808228a36a4399', 'detected': {'person': 0.904, 'skis': 0.5297}, 'caption': 'a couple of people riding skis down a snow covered slope'}, 142687: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000142687.jpg', 'image_sha256': 'd3e6cc889531dc150ff030d174a1ca497590a2c31877a46c59c56cb5768a1d15', 'detected': {'person': 0.9179, 'pizza': 0.7696, 'oven': 0.7397}, 'caption': 'a man standing in front of a pizza oven'}, 154816: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000154816.jpg', 'image_sha256': '9adf20cc383c8fb52672a473369b56e128d3beb260564224dfce83839a76b926', 'detected': {'person': 0.9469, 'baseball glove': 0.8354, 'car': 0.7515}, 'caption': 'a young boy swinging a baseball bat at a ball'}, 161187: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000161187.jpg', 'image_sha256': '9795d7defbf662062df88c905bda6f249cc8e4c0d0ef5869c9bfeb9f6c9358cf', 'detected': {'train': 0.9187}, 'caption': 'a black train engine traveling down train tracks'}, 161978: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000161978.jpg', 'image_sha256': 'e63efa78dc3ef4bc5490566bfac5a21b3669e1ec12f3dc700462cd6c196d46ef', 'detected': {'person': 0.9383, 'skateboard': 0.8877}, 'caption': 'a group of people riding skateboards at a skate park'}, 162580: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000162580.jpg', 'image_sha256': '7d8d43bbadfe446b2a556c0c5c6d0d432f8288c482d7182390f7a1ab5cbd86f8', 'detected': {'person': 0.8984, 'bench': 0.5006}, 'caption': 'a boy sitting on a chair holding a teddy bear'}, 163112: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000163112.jpg', 'image_sha256': '7fd78aea0d189069e19f2eeb275c4b6588fc1ef700c3a9bb5ceec41fe7be55dc', 'detected': {'person': 0.9284, 'bench': 0.8711}, 'caption': 'a man sitting on a bench in the dark'}, 163543: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000163543.jpg', 'image_sha256': '2699b7cdf58e47eacd0eaf20bd379e69d38f93de5c2fbfd0971933f4c1c1d0ad', 'detected': {'person': 0.8745, 'skis': 0.5972}, 'caption': 'a man riding skis down a snow covered slope'}, 166624: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000166624.jpg', 'image_sha256': 'a2510519a56ca14b46e416cdceb6ba3329b81dc9f34eeef1989ae738de2bf189', 'detected': {'person': 0.9346, 'skis': 0.6635}, 'caption': 'a person riding skis down a snow covered slope'}, 167656: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000167656.jpg', 'image_sha256': '751be9f6e7aa02888cd1722d50b602f791f85d6412ac41c0330353fcfa2e4bdc', 'detected': {'person': 0.8048, 'clock': 0.7516}, 'caption': 'a man standing on a tennis court holding a racquet'}, 189248: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000189248.jpg', 'image_sha256': '5dac625491c2928dcec8acd40dec5008cc6b5e23ae97af398fa55d6aa83d5edd', 'detected': {'person': 0.865, 'surfboard': 0.7217}, 'caption': 'a group of people riding boards on top of a wave'}, 197997: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000197997.jpg', 'image_sha256': '156be9f7cb2f8c3d437775925608aa12b4f47075f6be9b97fe3e8ddc0506327d', 'detected': {'bus': 0.9418, 'bicycle': 0.7592}, 'caption': 'an old blue bus parked on the side of a road'}, 199017: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000199017.jpg', 'image_sha256': '45be9bd59986b301f0d43e19ca287ad7b5edff50b67f43185135ccd28482ec53', 'detected': {'elephant': 0.8946}, 'caption': 'two people riding an elephant across a river'}, 199977: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000199977.jpg', 'image_sha256': 'c5d981fedb47d7cbf1608c9196ed6302617cbda171d6e6a18077f5df32a2a4e9', 'detected': {'airplane': 0.9569}, 'caption': 'a plane flying through a blue sky with smoke coming from the back'}, 202228: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000202228.jpg', 'image_sha256': 'f21ab0c8495622bb8dee961d19a9cf07ce15195e1ef65c52a954c9b3d6f5d606', 'detected': {'person': 0.8778, 'sink': 0.5755}, 'caption': 'a man taking a selfie in a bathroom mirror'}, 204757: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000204757.jpg', 'image_sha256': 'd2cca8b6bbe2de3919dc26cbe97353b27c270f0919ecbc66f1b22444dafb46bf', 'detected': {'person': 0.5547}, 'caption': 'a person holding a cell phone in their hand'}, 207561: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000207561.jpg', 'image_sha256': 'b4fb519dcd548355cf5be309860864777690d05230b5198201d8d0ae8a0bbcef', 'detected': {'person': 0.8831, 'surfboard': 0.5121}, 'caption': 'a group of people riding boards on top of a wave'}, 210394: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000210394.jpg', 'image_sha256': 'd3a536af8607a677c7bfd4c87d8a21a4e42402980635eef8b54d476317471c99', 'detected': {'bicycle': 0.8961, 'person': 0.8652, 'motorcycle': 0.7743, 'truck': 0.7676}, 'caption': 'a bus driving down a street next to a group of people'}, 214604: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000214604.jpg', 'image_sha256': 'ec20307b202bf738b10ec8e1711d7fa2d20902467d869a30419ade29b658179f', 'detected': {'person': 0.9194, 'baseball glove': 0.8096, 'baseball bat': 0.5338, 'bench': 0.5202}, 'caption': 'a baseball player holding a bat on a field'}, 217622: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000217622.jpg', 'image_sha256': 'ba17708fd73b1e4286c2a24fb9647ab97c5d5d44d568b5e9c5f710b45dc8d643', 'detected': {'person': 0.8888}, 'caption': 'a person on a surfboard riding a wave'}, 226008: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000226008.jpg', 'image_sha256': 'e8aa62990f1887bc2fc81a27f50f5b7dae8533d24bda58ae87e00882801f9415', 'detected': {'person': 0.8932}, 'caption': 'a woman riding a jet ski on a lake'}, 231079: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000231079.jpg', 'image_sha256': 'f244223156b9fa3173ab38b0b4a1383e8050fb6ed6153ec44bab610bebee5a84', 'detected': {'person': 0.9208}, 'caption': 'a man riding a wave on a surfboard'}, 236166: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000236166.jpg', 'image_sha256': 'd0f0b38251198a84cc2f1fb9ed30ab8461aa2c5b71cf59b43b5e9efa0ddaca2a', 'detected': {'person': 0.9014, 'dog': 0.8805}, 'caption': 'a man in a hat holding a dog'}, 259690: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000259690.jpg', 'image_sha256': 'd5296545bda4a910e90c0e252423bac9d61d11802d699ab2c27c562b7d00bd40', 'detected': {'person': 0.9355, 'baseball bat': 0.8378}, 'caption': 'a man holding a baseball bat next to a girl in a wheel chair'}, 263251: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000263251.jpg', 'image_sha256': '8758d20e48542cb02e65d4eec63924dd78ad53ae1baec4871d956586eed97e7f', 'detected': {'person': 0.9461, 'sports ball': 0.8141, 'umbrella': 0.7369}, 'caption': 'a man holding a pink umbrella over his head'}, 266491: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000266491.jpg', 'image_sha256': 'bceaea14275dbc3767a8f3a7c4cd0d2ccd71bc236898fd148ae318af639717ee', 'detected': {'person': 0.9319}, 'caption': 'two people cross country skiing in the woods'}, 270097: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000270097.jpg', 'image_sha256': '09949a0ed2014b00658fb9690d1b53b736a5c0f042d12f71d5a9cf6527f02906', 'detected': {'person': 0.9356, 'book': 0.6236}, 'caption': 'a man sitting in front of a mirror with a laptop'}, 271681: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000271681.jpg', 'image_sha256': '711a5cf3fafedfe15e46dbee827f8e1bc6a5f6ad9fb08c63daf856afe6841c3b', 'detected': {'person': 0.942, 'donut': 0.5911, 'car': 0.5779}, 'caption': 'a woman holding a piece of food in her hand'}, 276693: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000276693.jpg', 'image_sha256': '4b13991717d3eeade2896744ba28e3e909b182c0e829b37f386af011d1031eb4', 'detected': {'dog': 0.9312, 'person': 0.8576}, 'caption': 'a person walking a dog in the snow'}, 294537: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000294537.jpg', 'image_sha256': '1863125b46e51a5222f66976779315c8b9856f465f08fc81903c6983cc17dbc7', 'detected': {'person': 0.9299, 'skis': 0.7583, 'dog': 0.6077, 'backpack': 0.587}, 'caption': 'a man standing next to two children on skis'}, 296825: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000296825.jpg', 'image_sha256': 'c42e8035c0408d770b489931b4ca2eaaf6c9858b420647e19d198ec14ecdde4b', 'detected': {'truck': 0.9152, 'person': 0.9085}, 'caption': 'a yellow and red truck parked in a parking lot'}, 304596: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000304596.jpg', 'image_sha256': '13642708be873bdf664e3fbbefacdb1af816724e544079b295e51f3aba2c5000', 'detected': {'person': 0.9582, 'skateboard': 0.6427}, 'caption': 'a person jumping up in the air with a tennis racket'}, 308362: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000308362.jpg', 'image_sha256': 'b2ae19cdcd8e7fea0d1a3eae1e1b99f1afb2516e1cdb835d861c9805b17155b0', 'detected': {'bus': 0.9566}, 'caption': 'a bus parked in front of a building'}, 310369: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000310369.jpg', 'image_sha256': 'fc20b5cfe5b7acb9cef7b50340623dc5ca13490668fd4858fd9852ad6b1bf383', 'detected': {'person': 0.9292, 'chair': 0.7341, 'pizza': 0.5888, 'vase': 0.5152}, 'caption': 'a group of women sitting on a couch eating pizza'}, 314493: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000314493.jpg', 'image_sha256': '218ac03c8e305ad9cb443581051c573e7ebb6eab42b08f67167fa219d8177e94', 'detected': {'airplane': 0.9621, 'person': 0.7323}, 'caption': 'a small airplane flying through a blue sky'}, 314757: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000314757.jpg', 'image_sha256': '1626fead44f6142d9e0c82fe5e00fbf2b585ad14d085aa0f26acbf13a35de7d1', 'detected': {'person': 0.9355, 'laptop': 0.9131, 'couch': 0.897, 'remote': 0.7448, 'chair': 0.6067, 'vase': 0.6054, 'potted plant': 0.5951}, 'caption': 'a man standing in a living room holding a nintendo wii game controller'}, 314996: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000314996.jpg', 'image_sha256': 'a1fffc94bfc0487d557557c14aec98a0c5a056bc5724f662a9243c5d65f25645', 'detected': {'surfboard': 0.6079, 'person': 0.5456}, 'caption': 'a woman standing on a beach holding a surfboard'}, 317188: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000317188.jpg', 'image_sha256': '3c82292310e4e844ce44a1f75e4a3b05308398c708d9fb34e40388aa3f5121b5', 'detected': {'person': 0.9489, 'cell phone': 0.8267}, 'caption': 'a woman in a pink sweater holding a kite'}, 317595: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000317595.jpg', 'image_sha256': '7b08e4d7ca19652376de5a38ddb5610bfdbd9e279e15a62ca348ff64093ed44a', 'detected': {'person': 0.9483, 'cup': 0.8856, 'bottle': 0.8011, 'sandwich': 0.6339}, 'caption': 'a group of men sitting at a table eating sandwiches'}, 322145: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000322145.jpg', 'image_sha256': '6e5201c184874622c7f9324cc9413fe07463933157b8ac240e3f4237e6978194', 'detected': {'person': 0.9202}, 'caption': 'a man riding skis down a snow covered slope'}, 327656: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000327656.jpg', 'image_sha256': '1a4cca3fb36d5d6643f957a04c493912c6ea6639c28eebd39a3cbfad30affb09', 'detected': {'truck': 0.9495, 'person': 0.8651}, 'caption': 'a woman sitting in front of a large truck'}, 328791: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000328791.jpg', 'image_sha256': '4234add3ce536c140b95595fe09d815ffa41d29064d71c06fbb1d449a7307cbf', 'detected': {'person': 0.912, 'bicycle': 0.8686, 'car': 0.6673}, 'caption': 'a group of people standing outside of a street'}, 328921: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000328921.jpg', 'image_sha256': 'a1ce42f4a26f3ce7e738e42401579afef76bfff18a2b63ddd55f93ffe9819975', 'detected': {'person': 0.8781, 'tie': 0.5996}, 'caption': 'a group of three statues with hats on'}, 334309: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000334309.jpg', 'image_sha256': '4ad554c26a5d46b3548b92872606180756fb24786b3ed20819cb38124d85c3c9', 'detected': {'person': 0.9521, 'hot dog': 0.8394, 'car': 0.7617}, 'caption': 'a man holding a cell phone up to his mouth'}, 336431: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000336431.jpg', 'image_sha256': '4c17b31e2ecc89d08d97cf856fd8b725e361693ff649e5f0111b82df5c00434b', 'detected': {'chair': 0.9496, 'person': 0.9198, 'toothbrush': 0.8292}, 'caption': 'a young boy in a red shirt holding a toothbrush'}, 352729: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000352729.jpg', 'image_sha256': '3987d705bfc9b23d9e6f2f44504124b1097f4af061db6407e9da32229e2cd2ef', 'detected': {'person': 0.9258}, 'caption': 'a man riding a wave on top of a surfboard'}, 365129: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000365129.jpg', 'image_sha256': 'db29c68f7ce4f3c8eb60c154a26af2dd938fe0ab0ee5f0a9d4ef217fc26b72af', 'detected': {'boat': 0.8704}, 'caption': 'a group of boats floating on top of a body of water'}, 366822: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000366822.jpg', 'image_sha256': '7fb7cffcf05cd409b74bf6faaeb2fac3a2db9ec4ed46fa332f43d02f716a744f', 'detected': {'person': 0.913, 'snowboard': 0.8957, 'backpack': 0.588}, 'caption': 'a group of people standing next to each other in the snow'}, 374829: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000374829.jpg', 'image_sha256': '2df3ccfdb9afdeee7cef6d4c7056c32ea4bad12b1a8ca11ca6f678c9855be1c2', 'detected': {'person': 0.9243, 'baseball bat': 0.8026, 'chair': 0.7914, 'baseball glove': 0.7131}, 'caption': 'a baseball player holding a baseball bat on a field'}, 376206: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000376206.jpg', 'image_sha256': 'da6dc5bbfa9e31edf7b4f86d4ac27329484ea2e3bfd472e1f13853ebb0c127e7', 'detected': {'person': 0.8898, 'surfboard': 0.5379}, 'caption': 'a man riding a wave on top of a surfboard'}, 376368: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000376368.jpg', 'image_sha256': 'ddb8ad9b30d741beea4bd09aaea9ee8b9a8fe5ef2f024b6fe75000230adac40f', 'detected': {'boat': 0.7249, 'person': 0.672}, 'caption': 'a boat floating down a river next to a bridge'}, 377715: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000377715.jpg', 'image_sha256': '9bb676f6c18835150ae4dc3579254dbad777b7a46a001493394c5a1e564b87e3', 'detected': {'person': 0.9415}, 'caption': 'a baseball player holding a bat on a field'}, 377999: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000377999.jpg', 'image_sha256': '997f4df838f9905c2f10aa876c65f78d62fddba74e6801f86ca5cb9a6bba8bac', 'detected': {'boat': 0.9278, 'bicycle': 0.9068, 'person': 0.753}, 'caption': 'a white sail boat sitting on top of a body of water'}, 379048: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000379048.jpg', 'image_sha256': 'd0ea514285b73bafac19e106dc6b54043b3d9f1ef69cb15c26c72ffbdcfe7363', 'detected': {'baseball bat': 0.9263}, 'caption': 'a baseball bat and a bat in a display case'}, 384635: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000384635.jpg', 'image_sha256': '37e552f0ff8b71f7763af991860ab09afb4d17cfeabe6bc7bc63e3936394b01b', 'detected': {'train': 0.9518, 'person': 0.5341}, 'caption': 'a yellow and black train traveling down train tracks'}, 389123: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000389123.jpg', 'image_sha256': '3c203eb6626c81ad81cb8077fa75420ad575206710161df9b3596307c9878f20', 'detected': {'person': 0.7792, 'surfboard': 0.5641}, 'caption': 'two people riding a wave on top of surfboards'}, 391463: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000391463.jpg', 'image_sha256': '5f99c7640abfd52cf7026c3f7f26888eb5adb0dba21583d10e59219a96747e9b', 'detected': {'person': 0.8783, 'surfboard': 0.7015}, 'caption': 'a man walking across a beach holding a surfboard'}, 409241: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000409241.jpg', 'image_sha256': '9277da4159ae3baa64489e9ac1ccedfc368e4be7b1317978485fece5ca3b2de2', 'detected': {'train': 0.8966, 'person': 0.7142}, 'caption': 'a black and red train on a track'}, 409346: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000409346.jpg', 'image_sha256': '00eb9ea48688ff976f7bf0fcd88f158121196d2c21d45ca316aa26bfcee0d652', 'detected': {'person': 0.9462, 'bowl': 0.7066}, 'caption': 'a couple of women standing next to a table'}, 425676: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000425676.jpg', 'image_sha256': 'e45912757d8f3f2808e5410b50b24dd368182bee335f2900e3583f3bdc91750c', 'detected': {'person': 0.9266, 'baseball glove': 0.6234, 'baseball bat': 0.5813}, 'caption': 'a baseball player holding a bat next to home plate'}, 434993: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000434993.jpg', 'image_sha256': 'e4b1c7efa4d09d6dcc36b510be24b0e8d0179659d8f1639241c4d6283e350a36', 'detected': {'person': 0.9102, 'motorcycle': 0.6556}, 'caption': 'a man riding a skateboard up the side of a bench'}, 435309: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000435309.jpg', 'image_sha256': '3083c107cdf6101912e3d40e84947fc02541b756500edc44f4cc6706a226402b', 'detected': {'cell phone': 0.9392, 'person': 0.7633}, 'caption': 'a person holding a red cell phone in their hand'}, 438907: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000438907.jpg', 'image_sha256': '8125799f8600936368dd0bab99f4f4817f3965e79b8c94af8f4bdda8ff1bafa2', 'detected': {'person': 0.9337, 'skateboard': 0.9108}, 'caption': 'a young man flying through the air while riding a skateboard'}, 439522: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000439522.jpg', 'image_sha256': '663545f71af556370c7cba7fd8010a665d00c0b477075562a3d7669c6d853ad3', 'detected': {'person': 0.9623}, 'caption': 'a woman standing in the snow holding ski poles'}, 442417: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000442417.jpg', 'image_sha256': '7eab8f58f1253e7fcb878939efa604c9cb3213be25f8252dd1d909c58ae3e433', 'detected': {'person': 0.9181}, 'caption': 'a man talking on a cell phone in a building'}, 461262: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000461262.jpg', 'image_sha256': '169ee6127e74801817bcecdac871d1772afaf49ba2d1dea39d5f974e6150eb57', 'detected': {'person': 0.9377, 'baseball glove': 0.8258}, 'caption': 'a baseball player holding a bat on a field'}, 463722: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000463722.jpg', 'image_sha256': '6f97a7c69ddd0ab36a07f206842ac9daed4f5ddd893288b43cfc786a38505ef1', 'detected': {'horse': 0.8777, 'person': 0.7648}, 'caption': 'a man holding a red flag on top of a horse'}, 467580: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000467580.jpg', 'image_sha256': '86ce1bb71da431ea10ff229306b1ba2f5f116d3eb549db6ce8ce13a81ed8352a', 'detected': {'person': 0.9257}, 'caption': 'a man riding a wave on top of a surfboard'}, 467791: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000467791.jpg', 'image_sha256': '5a871d300f4805df86acd1fa1ca5210af591ec3ee1f974da1ab2181e2328e43a', 'detected': {'person': 0.9435, 'baseball bat': 0.7065, 'baseball glove': 0.6762}, 'caption': 'a baseball player holding a bat on a field'}, 472878: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000472878.jpg', 'image_sha256': '1aa015cccc33a151156800d54628b7fbb61a4f2602ab94330aa3ec1c760c319d', 'detected': {'surfboard': 0.837, 'person': 0.6979}, 'caption': 'a person riding a surf board on a wave'}, 481670: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000481670.jpg', 'image_sha256': '1926544f88dae8a09a3789fbe0d5e78605192824a79cd8254a768119674b24b7', 'detected': {'person': 0.8934, 'skis': 0.5162}, 'caption': 'a group of people riding skis down a snow covered slope'}, 492677: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000492677.jpg', 'image_sha256': 'ee3cd7d5d081f85d494e0135366103856b626d1bb6808a0db80968bf9b267b0f', 'detected': {'person': 0.9211}, 'caption': 'a man standing in front of a store talking on a cell phone'}, 497855: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000497855.jpg', 'image_sha256': '481b8149a6551113afaf5a3c03b3caaaeaa9875908c06f225bb795501e39938c', 'detected': {'apple': 0.8486, 'banana': 0.8411, 'orange': 0.7482}, 'caption': 'a pile of fruit sitting next to a banana'}, 499400: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000499400.jpg', 'image_sha256': '5ce69265f3aac9a596e446fd3b617fdabb4a53e31e7693745fb84474e612f159', 'detected': {'surfboard': 0.9549, 'person': 0.9112}, 'caption': 'a group of men standing next to two surfboards'}, 505099: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000505099.jpg', 'image_sha256': '6174f5d75f00dcb636ba8884c97ae112a49869489a25524f037f27668419099a', 'detected': {'person': 0.9289, 'baseball glove': 0.8539, 'baseball bat': 0.6202}, 'caption': 'a baseball player holding a baseball bat on a field'}, 511379: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000511379.jpg', 'image_sha256': 'd3b524785ed17e5672850c82ebc0ed4002052f34e2b3f4a1d49c9e1fea05ca05', 'detected': {'clock': 0.5825}, 'caption': 'a clock on a pole next to a flag'}, 515444: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000515444.jpg', 'image_sha256': 'a2b96d4717ba2e89a2cddea560c58774834741bd31088ec48f613371f96959d2', 'detected': {'person': 0.936, 'chair': 0.8039, 'hot dog': 0.5946}, 'caption': 'a man eating a hot dog at a baseball game'}, 525439: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000525439.jpg', 'image_sha256': '023a7b68203f01cf177bc3888345194e23a47ba7cfefb2775da807da3f2388f2', 'detected': {'skateboard': 0.9617, 'person': 0.9337}, 'caption': 'a person standing on a skateboard on a ramp'}, 528276: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000528276.jpg', 'image_sha256': 'e3477388ea64d498521cbd71ac751784c307c29752f662208b1794e83df7b4f8', 'detected': {'person': 0.8657, 'skateboard': 0.6064}, 'caption': 'a man riding a skateboard next to a park'}, 538414: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000538414.jpg', 'image_sha256': '3acbb8f8c1a5add4a7e43d9c71839c618809f38f1f177f0923fd029f582b9288', 'detected': {'person': 0.9363, 'baseball glove': 0.8224, 'baseball bat': 0.5415}, 'caption': 'a baseball player swinging a bat on a field'}, 543402: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000543402.jpg', 'image_sha256': 'ac9cf92b89f3d4ad33ad327e30d31e4a4b92de631fefff193f357e259b56ae69', 'detected': {'chair': 0.8883, 'umbrella': 0.8862}, 'caption': 'a chair sitting next to a purple umbrella'}, 548209: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000548209.jpg', 'image_sha256': '43edad5d793f07254e5153734822594561a051e904f9a994428dcd3ad0fd5b93', 'detected': {'person': 0.9148, 'skis': 0.5054}, 'caption': 'two people riding skis down a snow covered slope'}, 548550: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000548550.jpg', 'image_sha256': '1e36c519e9b122d870f5791303b470052a1671bee8c384ab29617a53f21d116b', 'detected': {'hot dog': 0.8901, 'cup': 0.8691}, 'caption': 'a hot dog sitting on top of a white plate'}, 557596: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000557596.jpg', 'image_sha256': 'bd666229b89cf45adb02d25ed48fbc475b28e4ce0813c1e21bda22b0a81d43d5', 'detected': {'person': 0.9384, 'tennis racket': 0.8663, 'chair': 0.6682}, 'caption': 'a woman standing on a tennis court holding a racquet'}, 561745: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000561745.jpg', 'image_sha256': '219b69c4b4f2707d4b94fa473d7d9c894a50bb90417d6d15584451b97642a927', 'detected': {'person': 0.9015, 'baseball glove': 0.8283, 'baseball bat': 0.6897, 'sports ball': 0.6602}, 'caption': 'a baseball player standing next to home plate'}, 562994: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000562994.jpg', 'image_sha256': '737d743fc2c11a9bdf96decdec1a91dc35ea1a00a62e1cddf35fd6b310c4bff4', 'detected': {'person': 0.9327}, 'caption': 'a man standing on a tennis court holding a racquet'}, 563885: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000563885.jpg', 'image_sha256': '5a9384569d2d1d01d6a6e0859d69d22341a255aca7112dee87109f12629d0dfb', 'detected': {'couch': 0.9358, 'person': 0.9134}, 'caption': 'a woman laying on a couch holding a stuffed animal'}, 564355: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000564355.jpg', 'image_sha256': 'f5ad1f2e2ad23293ad3d234d215f58c452f72f0ef1a68b0add51b0988e5b2eed', 'detected': {'person': 0.9639}, 'caption': 'a man standing on a snow covered ski slope'}, 570594: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000570594.jpg', 'image_sha256': '0943c14aa01b0eb88d71609d90b35053be55e61f396bcd3ee7ce640d5991ea74', 'detected': {'elephant': 0.9532, 'person': 0.9387}, 'caption': 'a man standing in front of two elephants'}, 571384: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000571384.jpg', 'image_sha256': '0569591b7e439255fb141880dbf71a7cc1102e8da0772f7c7a3b57d1af215d3d', 'detected': {'bus': 0.9541, 'person': 0.9347, 'car': 0.8616, 'handbag': 0.5876}, 'caption': 'a group of people standing outside of a bus'}, 574108: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000574108.jpg', 'image_sha256': 'f3de4ccc9a2a977676185b50d3690a84adb1758ba64d978499c8375024d8841d', 'detected': {'person': 0.9003, 'skis': 0.8628, 'backpack': 0.8008}, 'caption': 'a person riding skis down a snow covered slope'}, 575349: {'image_path': '/kaggle/input/datasets/vuthetam/mscoco-2014/images/val2014/COCO_val2014_000000575349.jpg', 'image_sha256': 'e32e8cb338eb4735acc1e493cdbddff5cae7f90cee81c0f43f2c483d2f36676b', 'detected': {'person': 0.9417}, 'caption': 'a young boy sitting in a kitchen eating a doughnut'}}
CLIP={'id': 'openai/clip-vit-base-patch16', 'revision': '57c216476eefef5ab752ec549e440a49ae4ae5f3', 'files_sha256': {'merges.txt': '9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a', 'pytorch_model.bin': 'ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0', 'special_tokens_map.json': 'f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf', 'config.json': 'eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad', 'tokenizer.json': 'a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4', 'preprocessor_config.json': '910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd', 'tokenizer_config.json': '3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade', 'vocab.json': '3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69'}}
RECEIPT={'sha256': '80e1280b9eef216c5db421a738c7a5e0550bb34b1a62a2a13b2c8e76d40b1a24', 'updates': 1500, 'source_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'parent_sha256': '56a90b2223b3bb64e433a342d048389dc3e80693fa574c2c1543e31e39a4377b'}
VOCAB_SHA='cc5f48afa9e471c7a8fd9e30ac58df2e47a54fbf4c0e86b6e0e340006b1601fd'
PROMPT_PINS={'prompt_texts.json': '35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2', 'prompt_embeddings.npy': 'f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c'}


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('TEST/CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    if CHECKPOINT_PATH:
        weights=Path(CHECKPOINT_PATH)
        if not weights.is_file():
            raise FileNotFoundError(f'Mounted checkpoint not found: {weights}')
    else:
        from kaggle_secrets import UserSecretsClient
        try:
            token=UserSecretsClient().get_secret(SECRET_NAME).strip()
        except Exception:
            raise RuntimeError('Enable the named Kaggle API Secret with access to the source notebook') from None
        if not token:
            raise RuntimeError('The named Kaggle API Secret is empty')
        TOKENS.append(token)
        cli_env=os.environ.copy()
        for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
            cli_env.pop(key,None)
        run([sys.executable,'-m','pip','install','-q','--target',CLI,'kaggle==2.2.5'],300,cli_env)
        cli_env.update(KAGGLE_API_TOKEN=token,PYTHONPATH=str(CLI)+os.pathsep+str(SOURCE),PYTHONUNBUFFERED='1')
        downloaded=SOURCE/'scst_v94'
        downloaded.mkdir()
        run([sys.executable,'-u','-c','from kaggle.cli import main; main()',
             'kernels','output',SOURCE_VERSION,'-p',downloaded,
             '--file-pattern',r'(^|/)update1500/pilot_checkpoint[.]bin$'],900,cli_env)
        candidates=list(downloaded.rglob('pilot_checkpoint.bin'))
        if len(candidates)!=1:
            raise ValueError('Expected the SCST1500 checkpoint from exact v94')
        weights=candidates[0]
    with weights.open('rb') as stream:
        if hashlib.file_digest(stream,'sha256').hexdigest()!=RESUME_SHA:
            raise ValueError('Changed SCST1500 checkpoint')

    sys.path.insert(0,str(SOURCE))
    (SOURCE/'captioning').mkdir(exist_ok=True)
    (SOURCE/'captioning/__init__.py').touch()
    for path,name in MAPPING.items():
        if name in {'semantic_data.py','patch_model.py','patch_beam.py','patch_data.py','full_data.py','data.py','real_bank_data.py','cuda_runtime_guard.py'}:
            download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,HASHES[path],SOURCE/name)
    for path,digest in TRAIN_PINS.items():
        name='bank_layers.py' if path.endswith('/model.py') else Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+TRAIN_REVISION+'/'+path,digest,SOURCE/name)
    for path,digest in {'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46'}.items():
        target=SOURCE/path if path.startswith('captioning/') else SOURCE/Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/277851d0fbc4643ae7001bc7702dd0ce5c91d0f6/'+path,digest,target)

    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
         'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2'],300)
    sys.path.insert(0,str(DEPS))
    os.environ.update(TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1')
    import numpy as np
    import torch
    import torch.nn.functional as F
    from PIL import Image
    from huggingface_hub import snapshot_download
    from transformers import CLIPModel, CLIPProcessor
    from captioning.multiscore_reranking import rerank_multiscore
    from csa_patch_model import CSAPatchCaptioner
    from patch_beam import beam_search, check_paths, text_from_ids
    from semantic_data import record_arrays
    from real_bank_data import sha

    if torch.__version__ not in ('2.11.0+cu128','2.11.0+cu130') or not torch.cuda.is_available():
        raise RuntimeError('Use Kaggle GPU T4 with CUDA-enabled Torch 2.11.0')
    device=torch.device('cuda:0')
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False

    mount=Path(CACHE_ROOT)
    with zipfile.ZipFile(mount/'cache_metadata_zip.bin') as z:
        receipt_bytes=z.read('part0/scst_cache_receipt.json')
        if hashlib.sha256(receipt_bytes).hexdigest()!='3a2caeb5950518ee5d82ac399de0462b5b4d07eaca0e380f7a9f8c49d5395510':
            raise ValueError('Wrong SCST Part 0 receipt')
        cache=json.loads(receipt_bytes)['files_sha256']
        def member(name):
            raw=z.read('part0/'+name)
            if hashlib.sha256(raw).hexdigest()!=cache[name]:
                raise ValueError('Changed cache member '+name)
            return raw
        import io
        valid_ids=np.load(io.BytesIO(member('valid/ids.npy')),allow_pickle=False)
        raw_vocab=member('vocabulary.json')
        if hashlib.sha256(raw_vocab).hexdigest()!=VOCAB_SHA:
            raise ValueError('Wrong vocabulary for SCST1500')
        words=json.loads(raw_vocab)
    patchfile=mount/'part0_valid_patches.npy'
    if sha(patchfile)!=cache['valid/patches.npy']:
        raise ValueError('Changed VALID patches')
    patches=np.load(patchfile,mmap_mode='r',allow_pickle=False)
    index={int(cid):i for i,cid in enumerate(valid_ids)}
    if len(index)!=5000 or not SAVED.keys()<=index.keys():
        raise ValueError('Wrong VALID coverage')
    links,=PROMPT_TEST_ROOT.rglob('prompt_links.jsonl')
    if sha(links)!='e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0':
        raise ValueError('Wrong prompt175 links')
    texts_path=links.parent/'prompt_texts.json'
    vector_path=links.parent/'prompt_embeddings.npy'
    for path,digest in PROMPT_PINS.items():
        if sha(links.parent/path)!=digest:
            raise ValueError('Wrong prompt175 asset '+path)
    texts=json.loads(texts_path.read_text())
    vectors=np.load(vector_path,allow_pickle=False)
    with links.open(encoding='utf8') as stream:
        original={r['coco_id']:r for r in map(json.loads,stream) if r['group']=='val' and r['coco_id'] in SAVED}
    if set(original)!=set(SAVED):
        raise ValueError('Missing original VAL prompts')
    upt_path,=Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2').rglob('prompt_cache.json')
    if sha(upt_path)!='24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d':
        raise ValueError('Wrong UPT prompt source')
    upt=json.loads(upt_path.read_text(encoding='utf-8'))
    for name, text in {'prompt_adapter_data.py': '"""NumPy-only instance joins for the region-linked CSA prompt pilot."""\nfrom collections import Counter\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\n\nPROMPT_SHA = \'24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d\'\nOBJECT_CAP, RELATION_CAP, IOU = 8, 3, .5\n\n\ndef label(value):\n    if not isinstance(value, str) or not value.strip():\n        raise ValueError(\'Missing object label\')\n    return \' \'.join(value.lower().replace(\'_\', \' \').split())\n\n\ndef box(value):\n    b = np.asarray(value, dtype=np.float64)\n    if b.shape != (4,) or not np.isfinite(b).all() or (b < 0).any() or (b > 1).any() or (b[2:] <= b[:2]).any():\n        raise ValueError(\'Expected normalized positive xyxy box\')\n    return b\n\n\ndef overlap(a, boxes):\n    a = box(a)\n    if not len(boxes):\n        return np.empty(0)\n    boxes = np.stack([box(b) for b in boxes])\n    intersection = np.maximum(0, np.minimum(a[2:], boxes[:, 2:]) - np.maximum(a[:2], boxes[:, :2])).prod(1)\n    return intersection / ((a[2:] - a[:2]).prod() + (boxes[:, 2:] - boxes[:, :2]).prod(1) - intersection)\n\n\ndef detector_regions(objects, width, height, saved_boxes, kinds):\n    """Reconstruct the existing cache\'s confidence-sort/clip/cap before label joins."""\n    if width <= 0 or height <= 0 or len(kinds) != len(saved_boxes):\n        raise ValueError(\'Invalid ROI dimensions\')\n    ordered = sorted(enumerate(objects), key=lambda x: (-float(x[1][\'conf\']), x[0]))[:150]\n    if not ordered:\n        if kinds != [\'full_frame_fallback\'] or not np.allclose(saved_boxes, [[0, 0, width, height]], rtol=0, atol=1e-4):\n            raise ValueError(\'Changed fallback ROI\')\n        return []\n    if kinds != [\'predicted_roi\'] * len(ordered):\n        raise ValueError(\'Detector and ROI counts/kinds disagree\')\n    result = []\n    for i, (_, obj) in enumerate(ordered):\n        score = float(obj[\'conf\'])\n        if not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid detector confidence\')\n        b = np.clip(np.asarray(obj[\'bbox\'], np.float32), [0, 0, 0, 0], [width, height, width, height]).astype(np.float32)\n        if b.shape != (4,) or not np.isfinite(b).all() or not np.allclose(b, saved_boxes[i], atol=1e-4, rtol=0):\n            raise ValueError(\'Detector order/box differs from saved ROI\')\n        result.append(dict(roi=i, label=label(obj[\'label\']), bbox=box(b / [width, height, width, height]).tolist()))\n    return result\n\n\ndef geometry(subject, obj):\n    s, o = box(subject), box(obj)\n    sw, sh = s[2:] - s[:2]; ow, oh = o[2:] - o[:2]\n    sc, oc = (s[:2] + s[2:]) / 2, (o[:2] + o[2:]) / 2\n    intersection = np.maximum(0, np.minimum(s[2:], o[2:]) - np.maximum(s[:2], o[:2])).prod()\n    return [float(v) for v in ((oc[0]-sc[0])/sw, (oc[1]-sc[1])/sh, np.log(ow/sw), np.log(oh/sh),\n            intersection/(sw*sh+ow*oh-intersection), intersection/(sw*sh), intersection/(ow*oh), np.linalg.norm(oc-sc))]\n\n\ndef prompt_text(name):\n    # Use the established UPT article handling, including plural skis/scissors.\n    from captioning.upt_prompts import noun\n    return noun(name)\n\n\ndef join_entry(entry, regions):\n    """Only selected objects/relations are consumed; previews/refs/GT are ignored."""\n    if entry[\'mode\'] not in (\'object_relation\', \'object_only\', \'no_objects\'):\n        raise ValueError(\'Unknown prompt mode\')\n    objects = entry[\'objects\']; relations = entry[\'relations\']\n    if not isinstance(objects, list) or not isinstance(relations, list) or len(relations) > RELATION_CAP:\n        raise ValueError(\'Malformed or unapproved prompt caps\')\n    by_index = {}; instances = set(); mapped = {}; stats = Counter(objects=len(objects), relations=len(relations))\n    for obj in objects:\n        index = obj[\'index\']; instance = obj[\'instance_id\']; name = label(obj[\'label\']); b = box(obj[\'bbox\'])\n        score = float(obj[\'confidence\'])\n        if type(index) != int or index in by_index or not isinstance(instance, str) or not instance or instance in instances:\n            raise ValueError(\'Duplicate/invalid object identity\')\n        if not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid UPT object score\')\n        by_index[index] = obj; instances.add(instance)\n        candidates = [r for r in regions if r[\'label\'] == name]\n        values = overlap(b, [r[\'bbox\'] for r in candidates])\n        if not len(values) or values.max() < IOU:\n            stats[\'unmatched_objects\'] += 1; continue\n        best = np.flatnonzero(np.isclose(values, values.max(), rtol=0, atol=1e-8))\n        if len(best) != 1:\n            stats[\'ambiguous_objects\'] += 1; continue\n        j = int(best[0]); r = candidates[j]\n        mapped[index] = dict(instance_id=instance, upt_index=index, roi=r[\'roi\'], label=name, bbox=b.tolist(),\n            roi_bbox=r[\'bbox\'], IoU=float(values[j]), confidence=score, text=prompt_text(name))\n    collisions = Counter(r[\'roi\'] for r in mapped.values())\n    for index in list(mapped):\n        if collisions[mapped[index][\'roi\']] > 1:\n            del mapped[index]; stats[\'colliding_objects\'] += 1\n    endpoints = []; valid_relations = []; seen_pairs = set()\n    for r in relations:\n        s, o = r[\'subject_index\'], r[\'object_index\']\n        if type(s) != int or type(o) != int or s == o or s not in by_index or o not in by_index:\n            raise ValueError(\'Broken relation endpoints\')\n        if r[\'subject_instance\'] != by_index[s][\'instance_id\'] or r[\'object_instance\'] != by_index[o][\'instance_id\']:\n            raise ValueError(\'Relation instance IDs differ\')\n        if label(r[\'subject\']) != \'person\' or label(by_index[s][\'label\']) != \'person\' or label(r[\'object\']) != label(by_index[o][\'label\']):\n            raise ValueError(\'Wrong subject/object role labels\')\n        score, threshold = float(r[\'score\']), float(r[\'threshold\'])\n        if not np.isfinite([score, threshold]).all() or not 0 <= score <= 1 or not 0 <= threshold <= 1.01 or score < threshold:\n            raise ValueError(\'Rejected/nonfinite relation in selected list\')\n        from captioning.upt_prompts import relation_text\n        expected = relation_text(r[\'predicate\'], label(r[\'object\']))\n        if r[\'text\'] != expected:\n            raise ValueError(\'Relation text differs from locked UPT template\')\n        key = (s, r[\'predicate\'], o)\n        if key in seen_pairs:\n            raise ValueError(\'Duplicate relation\')\n        seen_pairs.add(key)\n        if s not in mapped or o not in mapped:\n            stats[\'unmatched_relations\'] += 1; continue\n        endpoints.extend((s, o))\n        valid_relations.append(dict(subject_instance=r[\'subject_instance\'], object_instance=r[\'object_instance\'],\n            subject_roi=mapped[s][\'roi\'], object_roi=mapped[o][\'roi\'], text=expected, predicate=r[\'predicate\'],\n            score=score, threshold=threshold, geometry=geometry(by_index[s][\'bbox\'], by_index[o][\'bbox\'])))\n    order = list(dict.fromkeys(endpoints))\n    order += [i for i in sorted(mapped, key=lambda i: (-mapped[i][\'confidence\'], i)) if i not in order]\n    selected = [mapped[i] for i in order[:OBJECT_CAP]]\n    stats.update(mapped_objects=len(mapped), kept_objects=len(selected), mapped_relations=len(valid_relations),\n                 capped_objects=max(0, len(mapped)-OBJECT_CAP))\n    return dict(source_mode=entry[\'mode\'], objects=selected, relations=valid_relations, counts=dict(stats))\n\n\ndef cohort(ids):\n    ids = [int(i) for i in ids]\n    if len(ids) != len(set(ids)) or len(ids) < 10000:\n        raise ValueError(\'Need unique Karpathy TRAIN IDs\')\n    return sorted(ids, key=lambda i: hashlib.sha256(f\'42:{i}\'.encode()).hexdigest())[:10000]\n\n\ndef inference_record(meta, linked):\n    """Do not copy captions, sequences or any supervision into prompt payloads."""\n    return dict(coco_id=int(meta[\'coco_id\']), filename=meta[\'filename\'], group=meta[\'group\'],\n                roi_count=len(meta[\'region_kinds\']), **linked)\n\n\ndef tensor_batch(records, embeddings, texts):\n    """Data-only packing shared by the future worker and neural smoke tests."""\n    lookup = {s: i for i, s in enumerate(texts)}\n    if len(lookup) != len(texts) or embeddings.shape != (len(texts), 512) or not np.isfinite(embeddings).all():\n        raise ValueError(\'Invalid prompt embedding cache\')\n    n = len(records)\n    out = dict(object_text=np.zeros((n,8,512),np.float32), object_roi=np.zeros((n,8),np.int64),\n        object_box=np.zeros((n,8,4),np.float32), object_valid=np.zeros((n,8),bool),\n        relation_text=np.zeros((n,3,512),np.float32), relation_pair=np.zeros((n,3,2),np.int64),\n        relation_geometry=np.zeros((n,3,8),np.float32), relation_valid=np.zeros((n,3),bool))\n    for i, row in enumerate(records):\n        if len(row[\'objects\']) > 8 or len(row[\'relations\']) > 3:\n            raise ValueError(\'Prompt slot overflow\')\n        for j, obj in enumerate(row[\'objects\']):\n            if not 0 <= obj[\'roi\'] < row[\'roi_count\']:\n                raise ValueError(\'Object ROI out of bounds\')\n            out[\'object_text\'][i,j]=embeddings[lookup[obj[\'text\']]];out[\'object_roi\'][i,j]=obj[\'roi\']\n            out[\'object_box\'][i,j]=box(obj[\'bbox\']);out[\'object_valid\'][i,j]=True\n        for j, rel in enumerate(row[\'relations\']):\n            pair=[rel[\'subject_roi\'],rel[\'object_roi\']]\n            if pair[0] == pair[1] or any(not 0 <= k < row[\'roi_count\'] for k in pair):\n                raise ValueError(\'Invalid relation pair\')\n            g=np.asarray(rel[\'geometry\'],np.float32)\n            if g.shape!=(8,) or not np.isfinite(g).all():raise ValueError(\'Invalid pair geometry\')\n            out[\'relation_text\'][i,j]=embeddings[lookup[rel[\'text\']]];out[\'relation_pair\'][i,j]=pair\n            out[\'relation_geometry\'][i,j]=g;out[\'relation_valid\'][i,j]=True\n    return out\n', 'captioning/upt_prompts.py': '"""UPT predictions -> instance-linked text; no annotations or reference captions."""\nimport math\n\nfrom .selected_relation_prompts import entity_id, phrase\nfrom .upt_action_text import ACTION_TEXT, action_predicate\n\nMETHOD = \'upt_object_relation_prompt_v1\'\nPREDICATES = (\'holding\', \'riding\')\n\n\ndef noun(label):\n    return label if label in (\'skis\', \'scissors\') else phrase(label)\n\n\ndef relation_text(predicate, obj):\n    verb = {\'holding\': \'hold\', \'riding\': \'ride\'}.get(predicate, predicate)\n    if verb not in ACTION_TEXT:\n        raise ValueError(\'Unsupported prompt predicate.\')\n    # Do not infer surfing/skiing from object class alone. Keep UPT\'s broad ride.\n    if predicate == \'riding\' and obj == \'skis\':\n        return \'a person on skis\'\n    if verb in (\'jump\', \'run\'):\n        # These labels do not specify the object\'s grammatical role; do not invent one.\n        return f\'a person and {noun(obj)} (interaction: {ACTION_TEXT[verb]})\'\n    return f\'a person {ACTION_TEXT[verb]} {noun(obj)}\'\n\n\ndef from_audit(row):\n    """Allowlist inference fields; deliberately discard every GT/matched ID/label."""\n    detections = [{k: d[k] for k in (\'label\', \'confidence\', \'box\')} for d in row[\'detection_details\']]\n    claims = [{k: c[k] for k in (\'subject\', \'object\', \'predicate\', \'score\')} for c in row[\'candidates\']]\n    return detections, claims\n\n\ndef from_cache(row, vocabulary, all_actions=False):\n    h, w = row[\'resized_size_hw\']\n    if h <= 0 or w <= 0 or len(row[\'boxes_resized_xyxy\']) != len(row[\'labels\']) or len(row[\'labels\']) != len(row[\'object_scores\']):\n        raise ValueError(\'Invalid cache detection dimensions.\')\n    detections = []\n    for box, label, score in zip(row[\'boxes_resized_xyxy\'], row[\'labels\'], row[\'object_scores\']):\n        if not 0 <= label < len(vocabulary[\'objects\']):\n            raise ValueError(\'Invalid cache object label.\')\n        detections.append({\'label\': vocabulary[\'objects\'][label].replace(\'_\', \' \'), \'confidence\': score,\n                           \'box\': [max(0., min(1., x / scale)) for x, scale in zip(box, (w, h, w, h))]})\n    if len(row[\'pairs\']) != len(row[\'native_scores\']):\n        raise ValueError(\'Invalid native score dimensions.\')\n    claims = []\n    for (s, o), scores in zip(row[\'pairs\'], row[\'native_scores\']):\n        if not 0 <= s < len(detections) or not 0 <= o < len(detections):\n            raise ValueError(\'Invalid cache pair indices.\')\n        verbs = vocabulary[\'verbs_by_object\'][row[\'labels\'][o]]\n        if len(scores) != len(verbs):\n            raise ValueError(\'Sparse native score row differs from ontology.\')\n        for verb, score in zip(verbs, scores):\n            name = vocabulary[\'verbs\'][verb]\n            if name == \'no_interaction\':\n                continue\n            if all_actions and name not in ACTION_TEXT:\n                raise ValueError(f\'Unknown UPT action: {name}\')\n            if all_actions or name in (\'hold\', \'ride\'):\n                claims.append({\'subject\': s, \'object\': o, \'predicate\': action_predicate(name),\n                               \'score\': score})\n    return detections, claims\n\n\ndef build_prompt(filename, detections, claims, policy, max_relations=3, object_threshold=.5, max_object_names=8):\n    if max_relations < 1 or max_object_names < 1 or not math.isfinite(object_threshold) or not 0 <= object_threshold <= 1:\n        raise ValueError(\'Invalid prompt options.\')\n    thresholds = {p: value[\'threshold\'] for p, value in policy[\'predicates\'].items()}\n    if not thresholds or any(p not in {action_predicate(v) for v in ACTION_TEXT} for p in thresholds):\n        raise ValueError(\'Unknown or empty policy predicates.\')\n    if any(not math.isfinite(t) or not 0 <= t <= 1.01 for t in thresholds.values()):\n        raise ValueError(\'Invalid locked policy threshold.\')\n    objects = []\n    for i, d in enumerate(detections):\n        label, confidence, box = d[\'label\'], d[\'confidence\'], d[\'box\']\n        if not isinstance(label, str) or not label.strip() or label != label.strip():\n            raise ValueError(\'Invalid object name.\')\n        if not math.isfinite(confidence) or not 0 <= confidence <= 1:\n            raise ValueError(\'Invalid object confidence.\')\n        if len(box) != 4 or not all(math.isfinite(v) for v in box) or not 0 <= box[0] < box[2] <= 1 or not 0 <= box[1] < box[3] <= 1:\n            raise ValueError(\'Invalid normalized predicted box.\')\n        objects.append({\'index\': i, \'instance_id\': entity_id(filename, label, box),\n                        \'label\': label, \'confidence\': confidence, \'bbox\': box})\n    unique = {}\n    for c in claims:\n        s, o, p, score = (c[k] for k in (\'subject\', \'object\', \'predicate\', \'score\'))\n        if p not in thresholds or s == o or not 0 <= s < len(objects) or not 0 <= o < len(objects):\n            raise ValueError(\'Invalid relation pair/predicate.\')\n        if objects[s][\'label\'] != \'person\' or not math.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid relation actor/score.\')\n        key = (s, p, o)\n        if key not in unique or score > unique[key][\'score\']:\n            unique[key] = {\'subject_index\': s, \'object_index\': o, \'subject_instance\': objects[s][\'instance_id\'],\n                           \'object_instance\': objects[o][\'instance_id\'], \'subject\': \'person\', \'object\': objects[o][\'label\'],\n                           \'predicate\': p, \'score\': score, \'threshold\': thresholds[p],\n                           \'text\': relation_text(p, objects[o][\'label\'])}\n    # Margin is only a fixed cap ordering; not calibrated cross-predicate confidence.\n    accepted = sorted((c for c in unique.values() if c[\'score\'] >= c[\'threshold\']),\n                      key=lambda c: (-(c[\'score\'] - c[\'threshold\']), c[\'subject_index\'], c[\'predicate\'], c[\'object_index\']))\n    selected = accepted[:max_relations]\n    endpoints = {c[k] for c in selected for k in (\'subject_index\', \'object_index\')}\n    kept = [d for d in objects if d[\'confidence\'] >= object_threshold or d[\'index\'] in endpoints]\n    names = list(dict.fromkeys(d[\'label\'] for d in sorted(kept, key=lambda d: (-d[\'confidence\'], d[\'index\']))))[:max_object_names]\n    object_prompt = \'a photo containing \' + \', \'.join(noun(n) for n in names) + \'.\' if names else \'\'\n    texts = list(dict.fromkeys(c[\'text\'] for c in selected))\n    relation_prompt = \'a photo showing \' + \'; \'.join(texts) + \'.\' if texts else \'\'\n    previews = sorted(unique.values(), key=lambda c: (-c[\'score\'], c[\'subject_index\'], c[\'predicate\'], c[\'object_index\']))[:3]\n    return {\'objects\': kept, \'relations\': selected, \'object_prompt\': object_prompt,\n            \'relation_prompt\': relation_prompt, \'prompt\': relation_prompt or object_prompt or \'a photo.\',\n            \'mode\': \'object_relation\' if selected else \'object_only\' if names else \'no_objects\',\n            \'above_threshold_claims\': len(accepted), \'capped_claims\': len(accepted) - len(selected),\n            \'candidate_preview\': [{**c, \'passes_locked_threshold\': c[\'score\'] >= c[\'threshold\']} for c in previews]}\n', 'captioning/upt_action_text.py': '"""Explicit text for all 116 positive HICO action labels; no new action inference."""\n\nACTION_TEXT = dict(line.split(\'=\', 1) for line in \'\'\'adjust=adjusting\nassemble=assembling\nblock=blocking\nblow=blowing toward\nboard=boarding\nbreak=breaking\nbrush_with=brushing with\nbuy=buying\ncarry=carrying\ncatch=catching\nchase=chasing\ncheck=checking\nclean=cleaning\ncontrol=controlling\ncook=cooking\ncut=cutting\ncut_with=cutting with\ndirect=directing\ndrag=dragging\ndribble=dribbling\ndrink_with=drinking from\ndrive=driving\ndry=drying\neat=eating\neat_at=eating at\nexit=exiting\nfeed=feeding\nfill=filling\nflip=flipping\nflush=flushing\nfly=flying\ngreet=greeting\ngrind=grinding on\ngroom=grooming\nherd=herding\nhit=hitting\nhold=holding\nhop_on=hopping on\nhose=hosing\nhug=hugging\nhunt=hunting\ninspect=inspecting\ninstall=installing\njump=jumping\nkick=kicking\nkiss=kissing\nlasso=lassoing\nlaunch=launching\nlick=licking\nlie_on=lying on\nlift=lifting\nlight=lighting\nload=loading\nlose=losing\nmake=making\nmilk=milking\nmove=moving\nopen=opening\noperate=operating\npack=packing\npaint=painting\npark=parking\npay=paying at\npeel=peeling\npet=petting\npick=picking\npick_up=picking up\npoint=pointing\npour=pouring from\npull=pulling\npush=pushing\nrace=racing\nread=reading\nrelease=releasing\nrepair=repairing\nride=riding\nrow=rowing\nrun=running\nsail=sailing\nscratch=scratching\nserve=serving\nset=setting\nshear=shearing\nsign=signing\nsip=sipping from\nsit_at=sitting at\nsit_on=sitting on\nslide=sliding on\nsmell=smelling\nspin=spinning\nsqueeze=squeezing\nstab=stabbing\nstand_on=standing on\nstand_under=standing under\nstick=sticking\nstir=stirring\nstop_at=stopping at\nstraddle=straddling\nswing=swinging\ntag=tagging\ntalk_on=talking on\nteach=teaching\ntext_on=texting on\nthrow=throwing\ntie=tying\ntoast=toasting with\ntrain=training\nturn=turning\ntype_on=typing on\nwalk=walking\nwash=washing\nwatch=watching\nwave=waving at\nwear=wearing\nwield=wielding\nzip=zipping\'\'\'.splitlines())\n\nassert len(ACTION_TEXT) == 116 and \'no_interaction\' not in ACTION_TEXT\n\n\ndef action_predicate(verb):\n    return {\'hold\': \'holding\', \'ride\': \'riding\'}.get(verb, verb)\n\n\ndef uniform_policy(threshold=.5):\n    import math\n    if not math.isfinite(threshold) or not 0 < threshold <= 1:\n        raise ValueError(\'Exploratory threshold must be in (0,1].\')\n    return {\'mode\': \'all_actions_exploratory\', \'calibrated\': False,\n            \'predicates\': {action_predicate(v): {\'threshold\': threshold} for v in ACTION_TEXT}}\n', 'captioning/selected_relation_prompts.py': '"""Select scored detector claims without consulting annotation IDs or GT labels."""\nimport hashlib\nimport json\nimport math\nfrom collections import defaultdict\nfrom captioning.supervised_relations import RELATIONS\n\n\ndef entity_id(filename, label, box):\n    payload = json.dumps([filename, label, box], separators=(\',\', \':\'))\n    return \'det_\' + hashlib.sha256(payload.encode()).hexdigest()[:12]\n\n\ndef phrase(label):\n    article = \'an\' if label[:1].lower() in \'aeiou\' else \'a\'\n    return f\'{article} {label}\'\n\n\ndef build_selected_prompts(scored_pairs, thresholds, max_relations=3, object_inventory=None):\n    if len(thresholds) != len(RELATIONS) or max_relations < 1:\n        raise ValueError(\'Invalid thresholds/relation limit.\')\n    if not all(math.isfinite(x) for x in thresholds):\n        raise ValueError(\'Thresholds must be finite.\')\n    groups = defaultdict(list)\n    for filename in object_inventory or {}:\n        groups[filename] = []\n    for row in scored_pairs:\n        if len(row[\'scores\']) != len(RELATIONS) or not all(math.isfinite(v) for v in row[\'scores\']):\n            raise ValueError(\'Invalid scores.\')\n        for role in (\'subject\', \'object\'):\n            box = row[role+\'_box\']\n            if (len(box) != 4 or not all(math.isfinite(v) for v in box)\n                    or not 0 <= box[0] < box[2] <= 1 or not 0 <= box[1] < box[3] <= 1):\n                raise ValueError(\'Expected normalized nonempty detector boxes.\')\n        groups[row[\'filename\']].append(row)\n    result = {}\n    for filename, rows in sorted(groups.items()):\n        objects, candidates = {}, {}\n        for entity in (object_inventory or {}).get(filename, []):\n            identity = entity_id(filename, entity[\'label\'], entity[\'bbox\'])\n            objects[identity] = {**entity, \'instance_id\': identity,\n                                 \'prompt\': \'a photo of \'+phrase(entity[\'label\'])}\n        for row in rows:\n            ids = []\n            for role in (\'subject\', \'object\'):\n                box, label = row[role+\'_box\'], row[role]\n                identity = entity_id(filename, label, box)\n                ids.append(identity)\n                objects[identity] = {\'instance_id\': identity, \'label\': label, \'bbox\': box,\n                    \'confidence\': row.get(role+\'_detector_confidence\'), \'prompt\': \'a photo of \'+phrase(label)}\n            for c, relation in enumerate(RELATIONS):\n                score = row[\'scores\'][c]\n                if score < thresholds[c] or ids[0] == ids[1]:\n                    continue\n                key = (ids[0], relation, ids[1])\n                claim = {\'subject_instance\': ids[0], \'subject\': row[\'subject\'],\n                         \'object_instance\': ids[1], \'object\': row[\'object\'], \'relation\': relation,\n                         \'score\': score, \'threshold\': thresholds[c], \'score_margin\': score-thresholds[c],\n                         \'prompt\': f\'a photo of {phrase(row["subject"])} {relation.replace("_", " ")} {phrase(row["object"])}\'}\n                if key not in candidates or score > candidates[key][\'score\']:\n                    candidates[key] = claim\n        # Margin is a deterministic cap priority, NOT calibrated cross-class confidence.\n        ordered = sorted(candidates.values(), key=lambda c: (-c[\'score_margin\'], c[\'subject_instance\'],\n                                                            c[\'relation\'], c[\'object_instance\']))\n        selected = ordered[:max_relations]\n        labels = sorted({entity[\'label\'] for entity in objects.values()})\n        object_text = \'a photo of \'+\', \'.join(phrase(label) for label in labels) if labels else \'\'\n        relation_text = (\'a photo showing \'+\', \'.join(\n            f\'{phrase(c["subject"])} {c["relation"].replace("_", " ")} {phrase(c["object"])}\' for c in selected))\n        result[filename] = {\'objects\': list(objects.values()), \'relations\': selected,\n                            \'object_prompt\': object_text,\n                            \'relation_prompt\': relation_text if selected else \'\',\n                            \'prompt\': relation_text if selected else object_text,\n                            \'mode\': \'object_relation\' if selected else \'object_only\' if objects else \'no_objects\',\n                            \'above_threshold_claims\': len(ordered), \'capped_claims\': len(ordered)-len(selected)}\n    return result\n', 'captioning/supervised_relations.py': '"""V-COCO pair supervision. Missing links are UNKNOWN, never invented negatives."""\nimport math\nimport random\nfrom collections import Counter, defaultdict\n\nRELATION_SPECS = (\n    (\'hold\', \'obj\', \'holding\'), (\'ride\', \'instr\', \'riding\'),\n    (\'sit\', \'instr\', \'sitting_on\'), (\'carry\', \'obj\', \'carrying\'),\n    (\'eat\', \'obj\', \'eating\'), (\'talk_on_phone\', \'instr\', \'talking_on\'),\n)\nRELATIONS = [spec[2] for spec in RELATION_SPECS]\n\n\ndef decode_action(action):\n    """Official serialization stores all agents, then each role column."""\n    n = len(action[\'ann_id\'])\n    roles = action[\'role_name\']\n    flat = action[\'role_object_id\']\n    if n == 0 or len(flat) != n * len(roles):\n        raise ValueError(\'Invalid V-COCO role array.\')\n    if len(action[\'image_id\']) != n or len(action[\'label\']) != n:\n        raise ValueError(\'V-COCO column lengths differ.\')\n    if roles[0] != \'agent\' or flat[:n] != action[\'ann_id\']:\n        raise ValueError(\'Agent column does not match annotation IDs.\')\n    rows = {}\n    for i, sid in enumerate(action[\'ann_id\']):\n        if sid in rows or action[\'label\'][i] not in (-1, 0, 1):\n            raise ValueError(\'Duplicate actor or invalid action label.\')\n        rows[sid] = {\'image_id\': action[\'image_id\'][i], \'label\': action[\'label\'][i],\n                     \'roles\': {role: flat[j * n + i] for j, role in enumerate(roles)}}\n    return rows\n\n\ndef normalized_box(annotation, image):\n    x, y, w, h = annotation[\'bbox\']\n    width, height = image[\'width\'], image[\'height\']\n    if not all(math.isfinite(v) for v in (x, y, w, h, width, height)):\n        raise ValueError(\'Non-finite box.\')\n    if min(w, h, width, height) <= 0:\n        raise ValueError(\'Empty box/image.\')\n    box = [max(0, x / width), max(0, y / height),\n           min(1, (x + w) / width), min(1, (y + h) / height)]\n    if not (box[0] < box[2] and box[1] < box[3]):\n        raise ValueError(\'Box outside image.\')\n    return box\n\n\ndef build_pairs(actions, instances, caption_images, max_objects=12):\n    if max_objects < 1:\n        raise ValueError(\'max_objects must be positive.\')\n    by_action = {a[\'action_name\']: decode_action(a) for a in actions}\n    if any(name not in by_action for name, _, _ in RELATION_SPECS):\n        raise ValueError(\'Required V-COCO action missing.\')\n    images = {x[\'id\']: x for x in instances[\'images\']}\n    anns = {x[\'id\']: x for x in instances[\'annotations\']}\n    categories = {x[\'id\']: x[\'name\'] for x in instances[\'categories\']}\n    caption = {x[\'cocoid\']: x for x in caption_images}\n    if len(caption) != len(caption_images):\n        raise ValueError(\'Duplicate caption image ID.\')\n    by_image = defaultdict(list)\n    boxes = {}\n    for aid, ann in anns.items():\n        if ann.get(\'iscrowd\', 0):\n            continue\n        boxes[aid] = normalized_box(ann, images[ann[\'image_id\']])\n        by_image[ann[\'image_id\']].append(aid)\n    subjects = sorted(set(s for a in by_action.values() for s in a))\n    rows, stats = [], Counter()\n    for sid in subjects:\n        if sid not in anns:\n            raise ValueError(f\'Missing actor annotation {sid}.\')\n        ann = anns[sid]\n        iid = ann[\'image_id\']\n        if iid not in caption:\n            raise ValueError(f\'Image {iid} missing from caption split metadata.\')\n        if caption[iid][\'split\'] not in {\'train\', \'restval\'}:\n            stats[\'excluded_caption_heldout_subjects\'] += 1\n            continue\n        if sid not in boxes:\n            stats[\'excluded_crowd_subjects\'] += 1\n            continue\n        if categories[ann[\'category_id\']] != \'person\':\n            raise ValueError(\'V-COCO agent must be a person.\')\n        positives = set()\n        for name, role, _ in RELATION_SPECS:\n            item = by_action[name].get(sid)\n            if item and item[\'image_id\'] != iid:\n                raise ValueError(\'Actor/image ID mismatch.\')\n            if item and item[\'label\'] == 1:\n                oid = item[\'roles\'].get(role, 0)\n                if oid:\n                    if oid not in anns or anns[oid][\'image_id\'] != iid or oid == sid:\n                        raise ValueError(\'Invalid positive role object.\')\n                    if oid in boxes:\n                        positives.add(oid)\n                    else:\n                        stats[\'unusable_positive_box\'] += 1\n                else:\n                    stats[\'positive_action_without_visible_role\'] += 1\n        # Keep every positive; fill with deterministic same-image candidates.\n        candidates = sorted(set(by_image[iid]) - {sid} - positives)\n        candidates = sorted(positives) + candidates[:max(0, max_objects - len(positives))]\n        for oid in candidates:\n            labels = []\n            for name, role, _ in RELATION_SPECS:\n                item = by_action[name].get(sid)\n                label = -1\n                if item and item[\'label\'] == 0:\n                    label = 0  # Annotated absence of this actor action implies absence for any B.\n                elif item and item[\'label\'] == 1 and item[\'roles\'].get(role, 0) == oid:\n                    label = 1\n                # label=1 with another B, missing role, or label=-1 remains UNKNOWN.\n                labels.append(label)\n            if all(v == -1 for v in labels):\n                continue\n            image = caption[iid]\n            rows.append({\'image_id\': iid, \'filename\': image[\'filename\'],\n                         \'filepath\': image.get(\'filepath\', \'\'), \'subject_id\': sid,\n                         \'object_id\': oid, \'subject\': \'person\',\n                         \'object\': categories[anns[oid][\'category_id\']],\n                         \'subject_box\': boxes[sid], \'object_box\': boxes[oid],\n                         \'labels\': labels, \'caption_split\': image[\'split\']})\n    return rows, dict(stats)\n\n\ndef diagnostic_subset(rows, limit, seed=42):\n    if limit < 1:\n        raise ValueError(\'Pair limit must be positive.\')\n    rng = random.Random(seed)\n    pos = [r for r in rows if 1 in r[\'labels\']]\n    neg = [r for r in rows if 1 not in r[\'labels\']]\n    rng.shuffle(pos)\n    rng.shuffle(neg)\n    by_target = defaultdict(list)\n    for row in rows:\n        by_target[(row[\'image_id\'], row[\'object_id\'])].append(row)\n    # Enrich a bounded portion with confirmed actor-counterpart anchors; this is\n    # explicitly a development diagnostic, not a prevalence-preserving benchmark.\n    actor_anchors = [row for row in pos if any(\n        other[\'subject_id\'] != row[\'subject_id\']\n        and any(a == 1 and b == 0 for a, b in zip(row[\'labels\'], other[\'labels\']))\n        for other in by_target[(row[\'image_id\'], row[\'object_id\'])])]\n    chosen = actor_anchors[:limit // 8]\n    used = {(r[\'subject_id\'], r[\'object_id\']) for r in chosen}\n    chosen += [r for r in pos if (r[\'subject_id\'], r[\'object_id\']) not in used][\n        :max(0, min(len(pos), limit // 2) - len(chosen))]\n    used = {(r[\'subject_id\'], r[\'object_id\']) for r in chosen}\n    for anchor in list(chosen):\n        candidates = by_target[(anchor[\'image_id\'], anchor[\'object_id\'])]\n        for other in candidates:\n            key = (other[\'subject_id\'], other[\'object_id\'])\n            if (key not in used and other[\'subject_id\'] != anchor[\'subject_id\']\n                    and any(a == 1 and b == 0 for a, b in zip(anchor[\'labels\'], other[\'labels\']))):\n                chosen.append(other)\n                used.add(key)\n                break\n        if len(chosen) >= limit:\n            break\n    # Fill remaining slots with all-zero/unknown pairs before more positives.\n    # Positive-first fill previously skewed the intended diagnostic composition.\n    remaining = [r for r in neg + pos if (r[\'subject_id\'], r[\'object_id\']) not in used]\n    chosen += remaining[:max(0, limit - len(chosen))]\n    return sorted(chosen, key=lambda r: (r[\'image_id\'], r[\'subject_id\'], r[\'object_id\']))\n\n\ndef contrast_edges(rows, max_per_positive=4):\n    """Only annotated-zero claims can be negative; no role swaps inferred false."""\n    by_target = defaultdict(list)\n    for i, row in enumerate(rows):\n        by_target[(row[\'image_id\'], row[\'object_id\'])].append(i)\n    edges = []\n    for i, row in enumerate(rows):\n        for r, label in enumerate(row[\'labels\']):\n            if label != 1:\n                continue\n            wrong_pred = [(i, c, \'predicate\') for c, v in enumerate(row[\'labels\']) if v == 0]\n            wrong_actor = [(j, r, \'actor\') for j in by_target[(row[\'image_id\'], row[\'object_id\'])]\n                           if rows[j][\'subject_id\'] != row[\'subject_id\'] and rows[j][\'labels\'][r] == 0]\n            # Alternate types so a predicate-heavy anchor does not exclude all actor edges.\n            negatives = []\n            while wrong_pred or wrong_actor:\n                if wrong_actor:\n                    negatives.append(wrong_actor.pop(0))\n                if wrong_pred:\n                    negatives.append(wrong_pred.pop(0))\n            edges += [(i, r, j, c, kind) for j, c, kind in negatives[:max_per_positive]]\n    return edges\n\n\ndef pair_statistics(rows):\n    return {\'pairs\': len(rows), \'images\': len({r[\'image_id\'] for r in rows}),\n            \'labels\': {name: dict(Counter(str(r[\'labels\'][c]) for r in rows))\n                       for c, name in enumerate(RELATIONS)},\n            \'contrast_edges\': dict(Counter(e[-1] for e in contrast_edges(rows)))}\n\n\ndef select_unseen_pairs(rows, seen, count=500, seed=20261013):\n    eligible = [r for r in rows if r[\'image_id\'] not in seen]\n    selected = diagnostic_subset(eligible, count, seed=seed)\n    if len(selected) != count:\n        raise ValueError(\'Insufficient new pairs; do not silently change protocol.\')\n    if {r[\'image_id\'] for r in selected} & set(seen):\n        raise ValueError(\'Seen image selected.\')\n    return selected, eligible\n'}.items():
        path=SOURCE/name
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_text(text,encoding='utf-8')
    from captioning.upt_prompts import noun
    from prompt_adapter_data import geometry, label
    def direct_prompt(row, entry):
        objects = {obj['instance_id']: obj for obj in entry['objects']}
        relations = entry['relations']
        endpoints = list(dict.fromkeys(rel[field] for rel in relations
                                      for field in ('subject_instance', 'object_instance')))
        remaining = sorted((obj for obj in objects.values() if obj['instance_id'] not in endpoints),
                           key=lambda obj: (-obj['confidence'], obj['index']))
        order = (endpoints + [obj['instance_id'] for obj in remaining])[:8]
        if len(objects) != len(entry['objects']) or len(relations) > 3 or not set(endpoints) <= set(order):
            raise ValueError('Invalid UPT identities or prompt capacity')
        packed_objects = [dict(instance_id=i, roi=j, bbox=objects[i]['bbox'],
                               roi_bbox=objects[i]['bbox'], text=noun(label(objects[i]['label'])))
                          for j, i in enumerate(order)]
        packed_relations = [dict(subject_instance=r['subject_instance'], object_instance=r['object_instance'],
                                 text=r['text'], geometry=geometry(objects[r['subject_instance']]['bbox'],
                                                                  objects[r['object_instance']]['bbox']))
                            for r in relations]
        return dict(row, roi_count=len(order), objects=packed_objects, relations=packed_relations)

    def relation_keys(row):
        return {(r['subject_instance'],r['object_instance'],r['text']) for r in row['relations']}

    edited={cid:direct_prompt(row,upt[row['filename']]) for cid,row in original.items()}
    recovered={cid:len(relation_keys(edited[cid])-relation_keys(row)) for cid,row in original.items()}
    if (sum(recovered.values())!=207
            or any(not relation_keys(row)<=relation_keys(edited[cid]) for cid,row in original.items())):
        raise ValueError('Unexpected relation coverage on the locked VALID cohort')
    del upt
    print('Packed direct UPT prompts for 130 images. Restored 207 relation proposals.',flush=True)

    root=Path(snapshot_download(CLIP['id'],revision=CLIP['revision'],allow_patterns=list(CLIP['files_sha256'])))
    for name,digest in CLIP['files_sha256'].items():
        if sha(root/name)!=digest:
            raise ValueError('Changed CLIP asset '+name)
    clip=CLIPModel.from_pretrained(root,local_files_only=True,use_safetensors=False,attn_implementation='eager').float().to(device).eval()
    processor=CLIPProcessor.from_pretrained(root,local_files_only=True,use_fast=False)
    ck=torch.load(weights,map_location='cpu',weights_only=False)
    for field in ('updates','source_sha256','parent_sha256'):
        if ck[field]!=RECEIPT[field]:
            raise ValueError('Wrong checkpoint lineage')
    model=CSAPatchCaptioner(len(words),ck['model']['core.nouns'],ck['model']['core.verbs'])
    model.load_state_dict(ck['model'],strict=True)
    del ck
    model.to(device).eval()

    def encode_text(strings):
        t=processor.tokenizer(strings,padding=True,truncation=False,return_tensors='pt').to(device)
        if t.input_ids.shape[1]>77:
            raise ValueError('CLIP input too long')
        return F.normalize(clip.get_text_features(**t).float(),dim=-1)

    with torch.inference_mode():
        missing=sorted({o['text'] for row in edited.values() for o in row['objects']}
                       | {r['text'] for row in edited.values() for r in row['relations']})
        missing=[t for t in missing if t not in texts]
        if missing:
            vectors=np.concatenate((vectors,encode_text(missing).cpu().numpy()))
            texts+=missing

        def generate(row,x,image_vector,detected):
            arrays=record_arrays(row,texts)
            prompt={k:v for k,v in arrays.items() if k not in ('object_text_id','relation_text_id')}
            for role in ('object','relation'):
                prompt[role+'_text']=vectors[arrays[role+'_text_id']]*arrays[role+'_valid'][:,None]
            prompt={k:torch.from_numpy(np.ascontiguousarray(v[None])).to(device) for k,v in prompt.items()}
            memory=model.encode(x,prompt)
            seq,scores=beam_search(model,memory,width=5,maximum=20)
            check_paths(model,memory,seq,scores)
            tokens=seq[0].cpu().tolist()
            caps=[text_from_ids(v,words) for v in tokens]
            sim=(2.5*(image_vector*encode_text(caps)).sum(-1).clamp_min(0)).cpu().tolist()
            candidates=[dict(ids=i,caption=c,logprob=l,clipscore=s) for i,c,l,s in zip(tokens,caps,scores[0].cpu().tolist(),sim,strict=True)]
            picked,_=rerank_multiscore(dict(candidates=candidates),detected,.4,.3)
            return dict(candidates=candidates,rank=picked['original_rank'],caption=picked['caption'])

        pairs=[]
        for cid,old in original.items():
            if time.monotonic() >= PARENT_DEADLINE:
                raise TimeoutError('VALID inference deadline')
            saved=SAVED[cid]
            path=Path(saved['image_path'])
            if sha(path)!=saved['image_sha256']:
                raise ValueError('Changed source image')
            with Image.open(path) as im:
                pixels=processor(images=im.convert('RGB'),return_tensors='pt')['pixel_values'].to(device)
            image_vector=F.normalize(clip.get_image_features(pixel_values=pixels).float(),dim=-1)
            x=torch.from_numpy(np.array(patches[index[cid]:index[cid]+1],dtype=np.float32)).to(device)
            control=generate(old,x,image_vector,saved['detected'])
            candidate=generate(edited[cid],x,image_vector,saved['detected'])
            historical=saved['caption']
            pairs.append(dict(coco_id=cid,control=control,direct_upt=candidate,historical=historical,recovered_proposals=recovered[cid],
                              control_matches_history=control['caption']==historical))
            print(cid,control['caption'],' -> ',candidate['caption'],flush=True)
    (OUTPUT/'result.json').write_text(json.dumps(dict(images=130,split='val',recovered_proposals=sum(recovered.values()),checkpoint_sha256=RESUME_SHA,
        new_clip_texts=missing,records=pairs,changed=sum(r['control']['caption']!=r['direct_upt']['caption'] for r in pairs),
        control_matches_history=sum(r['control_matches_history'] for r in pairs)),indent=2),encoding='utf-8')
    complete=True
finally:
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ('result.json','notebook.log'):
            path=OUTPUT/name
            if path.is_file():z.write(path,name)
    print('SEND REVIEW:',review, 'complete:',complete,flush=True)
